# 03 — Fractional OU Estimation
Runs the fOU estimator on **all cointegrated pairs**. H is estimated unrestricted; the H<0.5 rule is applied later.

In [1]:

import pickle
import numpy as np
import pandas as pd

from src.fractional_OU import fit_cointegrated_pairs_fractional_ou

In [4]:
cointegrated_pairs = pd.read_parquet("data/processed/cointegrated_pairs.parquet")
with open("data/processed/spreads.pkl", "rb") as f:
    spreads = pickle.load(f)
if "pair" not in cointegrated_pairs.columns:
    cointegrated_pairs["pair"] = cointegrated_pairs["dependent"] + "-" + cointegrated_pairs["independent"]
print(f"cointegrated pairs: {len(cointegrated_pairs)}")
print(f"number of spreads: {len(spreads)}")
cointegrated_pairs.head()

cointegrated pairs: 446
number of spreads: 446


,dependent,independent,alpha,beta,adf,pvalue,pair
0,SHW,HD,-0.580495,1.084491,-5.474267,0.000002,SHW-HD
1,ORCL,ETN,0.640542,0.742037,-4.993938,0.000023,ORCL-ETN
2,SPGI,ICE,-1.358285,1.542537,-4.985110,0.000024,SPGI-ICE
3,ETN,JBL,1.246885,0.904839,-4.947990,0.000028,ETN-JBL
4,NDSN,ODFL,2.783512,0.528448,-4.928276,0.000031,NDSN-ODFL


In [6]:
pair_keys = set(zip(cointegrated_pairs["dependent"], cointegrated_pairs["independent"]))
spread_keys = set(spreads.keys())
print("Pairs only in dataframe:", len(pair_keys-spread_keys))
print("Spreads only in dictionary:", len(spread_keys-pair_keys))
assert pair_keys == spread_keys

Pairs only in dataframe: 0
Spreads only in dictionary: 0


In [7]:
fou_parameters = fit_cointegrated_pairs_fractional_ou(spreads, cointegrated_pairs)
print(f"Successfully estimated: {len(fou_parameters)}")
fou_parameters.head()

Successfully estimated: 446


,pair,dependent,independent,mu,kappa,sigma,hurst,variance,drift_half_life,n_obs
0,SHW-HD,SHW,HD,-1.124588e-14,0.030470,0.015142,0.479861,0.003215,22.748466,1759
1,ORCL-ETN,ORCL,ETN,5.619912e-15,0.024537,0.015696,0.480991,0.004293,28.248704,1759
2,SPGI-ICE,SPGI,ICE,5.583557e-15,0.009410,0.017839,0.425882,0.008012,73.660323,1759
3,ETN-JBL,ETN,JBL,1.195683e-15,0.017677,0.017089,0.484030,0.007166,39.211516,1759
4,NDSN-ODFL,NDSN,ODFL,3.778419e-15,0.002895,0.015797,0.363547,0.007994,239.454189,1759


In [8]:
print(fou_parameters["hurst"].describe(percentiles=[.05,.10,.25,.50,.75,.90,.95]))
print("\nH < 0.5:", (fou_parameters["hurst"] < 0.5).sum())
print("H >= 0.5:", (fou_parameters["hurst"] >= 0.5).sum())
print("Correlation H-kappa:", fou_parameters[["hurst","kappa"]].corr().iloc[0,1])
print("Correlation H-half-life:", fou_parameters[["hurst","drift_half_life"]].corr().iloc[0,1])

count    446.000000
mean       0.468459
std        0.063366
min        0.264904
5%         0.358006
10%        0.392174
25%        0.432232
50%        0.467748
75%        0.511019
90%        0.547391
95%        0.565031
max        0.699142
Name: hurst, dtype: float64

H < 0.5: 309
H >= 0.5: 137
Correlation H-kappa: 0.8037385195260727
Correlation H-half-life: -0.4390232385230256


In [9]:
OUTPUT ="data/processed/fractional_ou_parameters.parquet"
fou_parameters.to_parquet(OUTPUT,index=False)
print(f"Saved: {OUTPUT}")

Saved: data/processed/fractional_ou_parameters.parquet
